# Analysis Overview
**66 FINDINGS · 6 ANALYSIS AREAS · EXECUTIVE SUMMARY · FEATURE CANDIDATES**

---


> **Dataset (lf_all):** ~94.4M rows · 3 years (2023–2025) · 16 tram lines + Line E · Zurich  
> **Cleaned (lf_clean):** canceled==False · stop_sequence > 1 · no Nov/Dec 2025 · no Line E


## Inhalt

- [Executive Summary](#executive-summary)
- [Data Strategy — lf_clean Definition](#data-strategy--lf_clean-definition)
- [Notebooks](#notebooks)
- [Line Colors](#line-colors)
- [Analysis Result](#analysis-result)
  - [Core Questions](#core-questions)
  - [Key Findings](#key-findings)
  - [KPI Overview](#kpi-overview)
  - [Modelling Insights](#modelling-insights)


## Executive Summary

**94.4 Mio. Datenpunkte · 3 Jahre (2023–2025) · 16 Tramlinien · Zürich**

Das Zürcher Tramnetz fährt mit einer systemischen Schwäche: **an 71.5% aller Halte akkumuliert die Tram Verspätung** — kein Puffer ist eingebaut. OTP liegt bei 87%. Das ist stabil, aber strukturell fragil.

**Die 6 stärksten Erkenntnisse:**

| # | Kernbotschaft | Finding |
|:---|:---|:---|
| 1 | Verspätungen entstehen an der **Peripherie**, nicht im Zentrum — Central und Paradeplatz performen gut, Schwamendingen und Oerlikon (K11/K12) sind die Hotspots | F-SPAT-01/03 |
| 2 | **Kein Morgenrush** — das dominante Muster ist Feierabend + Events. Peak um 21h (67.9s) durch Abreisewellen. Donnerstag schlechtester Wochentag | F-TEMP-01/02 |
| 3 | **Schnee ist der stärkste Einzeleinflussfaktor** (+54s, OTP −10.9pp) — und geografisch klar trennbar von Regen (Höhenlagen vs. Flusstäler) | F-WEAT-01/07 |
| 4 | **Feiertage sind die besten Tage** (−9.9s) — der Berufsverkehrs-Rückgang überwiegt jeden Event-Effekt | F-EVNT-01 |
| 5 | Der **größte Fahrplanwechsel in VBZ-Geschichte** (Dez 2023) ist im Delay-Signal unsichtbar (+0.5s netzweit) — und zielte nicht auf die Problemkreise | F-NET-03/09 |
| 6 | **Vorhersagbar = Strukturell = Steuerbar** — MAE 18,56 s beweist: Delays folgen Mustern. Das Modell identifiziert WO und WANN Fahrplan-Puffer fehlen und liefert damit die Grundlage für datengetriebenes Schedule-Design | F-REC-01/02 |

→ Vollständige Findings: [Key Findings](#Key-Findings) · Feature-Kandidaten: [Modelling Insights](#Modelling-Insights)

---

**Kernthese — vier Befunde, eine Aussage:**

| Befund | Finding |
|:---|:---|
| 71.5% aller Halte akkumulieren Delay · 71.3% haben 0s dwell_time | F-TARGET-03 · F-SPAT-08 |
| Hotspots ausschliesslich an der Peripherie — zentrale Knoten performen gut | F-SPAT-01 · F-SPAT-07 |
| Netzausbau 2023 zielte auf gut-performende Kreise — Problemkreise K11/K12 erhielten nichts | F-NET-09 |
| MAE 18,56 s beweist Strukturalität: Zufällige Delays sind nicht so präzise vorhersagbar | F-REC-01 |

> **"Die Verspätungen im Zürcher Tramnetz sind vorhersagbar, sie sind systematisch —  
> und das Modell zeigt wo der Fahrplan geändert werden muss."**

## Data Strategy — lf_clean Definition



> Basis für alle Trend- und Modellanalysen. Begründung jedes Schritts.

| Filter-Schritt | Bedingung | Anteil entfernt | Begründung |
|:---|:---|:---|:---|
| Non-canceled | `canceled == False` | ~4.4% | Ausgefallene Fahrten haben keine validen Delay-Werte |
| Starthalte | `stop_sequence > 1` | ~4.8% der Non-canceled | Startstops zeigen künstlich niedrige `arrival_delay`-Werte (kein Vorgänger) |
| Linie E | `line_name != 'E'` | ~0.003% | Massiver Ausreisser (OTP 55.7%, 130s Ø Delay, nur 2'511 Zeilen) — separat behandeln |
| Nov/Dez 2025 | Monate 11–12 im Jahr 2025 entfernt | Artefakt | Fahrplanwechsel j25→j26 GTFS: delay_delta springt auf +17.1s/+26.0s — kein echter Betriebstrend |

**Effekt der Bereinigung auf Mittelwerte:** `arrival_delay` +1.1s · `delay_delta` −0.8s — minimal, Tendenzen unverändert.

**Gesamtzeilen:** lf_all = 93'904'623 · Non-canceled = 89'714'901 · lf_clean ≈ 85M (nach allen Filtern)

→ Details und Quantifizierung: `03_analysis_1-target.ipynb` (F-TARGET-05, F-TARGET-06, F-TARGET-12, F-TARGET-13)

### Analysis vs. Model — Data Basis

| | Analyse-Notebooks (`03_analysis_*`) | Modell (`06_prediction_*`) |
|:---|:---|:---|
| Datenbasis | `lf_all` = TRAIN + TEST · alle 3 Jahre | `train_final.parquet` · 2023–2024 |
| Warum | EDA darf alles sehen | Kein Blick in die Zukunft — 2025 bleibt unberührt |
| Zeilen | ~85 M (lf_clean) | 41.2M Train · 14.3M Val · ~25M Test |
| `departure_delay` | ✅ verfügbar — für Post-hoc-Analyse | ❌ ausgeschlossen — Leakage |
| `delay_delta` | ✅ verfügbar — zeigt Akkumulationsmuster | ❌ ausgeschlossen — Leakage |
| Sampling | Aggregationen: voller Datensatz via Polars LazyFrame. Korrelationen/Scatter: `gather_every(2)` + `sample(fraction=0.1)` → ~4.5M | Kein Sampling — LightGBM trainiert auf vollen 41.2M Zeilen |

Warum die Analyse alle 3 Jahre sieht: EDA beschreibt Realität — dafür soll sie so viel Daten wie möglich sehen. Der Train/Test-Split ist eine Modellierungs-Konvention die verhindert dass das Modell die Zukunft "kennt". Für die Analyse ist das irrelevant — eine Haltestelle die 2025 ein Hotspot ist, war es auch 2023 und 2024.

## Notebooks

| Notebook | Focus |
|:---|:---|
| `03_analysis_1-target.ipynb` | Delay distribution, OTP, arr vs dep, cancellations |
| `03_analysis_2-network.ipynb` | Netzveränderungen 2023–2025 · Vor/Nachher · Einlaufzeit · Hotspots · Versorgungsqualität |
| `03_analysis_3-temporal.ipynb` | Hour · weekday · month · season · full year |
| `03_analysis_4-spatial.ipynb` | Stops · districts · lines |
| `03_analysis_5-meteo.ipynb` | Rain · wind · snow · temperature |
| `03_analysis_6-events.ipynb` | Holidays · events · event size |

> **Hinweis für alle Notebooks:** Das Tramnetz hat sich im Analysezeitraum 2023–2025 verändert.
> Fahrplanwechsel Dezember 2023 (j23 → j24): Linien 9, 11 und 13 wurden fundamental umgebaut.
> Bei linienbezogenen Befunden immer `03_analysis_2-network.ipynb` als Kontext heranziehen.



## Line Colors



Offizielle VBZ-Linienfarben aus GTFS `routes.txt` — verfügbar via `line_color("12")` aus `zh_tram_flow.config`.

| Linie | Farbe | | Linie | Farbe | | Linie | Farbe |
|:---:|:---|:---|:---:|:---|:---|:---:|:---|
| **2** | `#E20A16` | | **8** | `#8AB51F` | | **14** | `#008DC5` |
| **3** | `#00892F` | | **9** | `#11296F` | | **15** | `#E20A16` |
| **4** | `#11296F` | | **10** | `#E12472` | | **17** | `#8E224D` |
| **5** | `#734522` | | **11** | `#00892F` | | **19** | `#E20A16` |
| **6** | `#CA7D3C` | | **12** | `#92D6E3` | | **E** | `#E20A16` |
| **7** | `#000000` | | **13** | `#FFCC00` | | | |


## Analysis Result


### Core Questions


1. Where do delays occur? — stops, districts, lines
2. When do delays occur? — time of day, weekday, season
3. What amplifies delays? — weather, events
4. What does the target itself look like? — distribution, OTP, arr vs dep
5. Which features correlate most with delays?
6. Can delays be predicted? → modeling

### Key Findings




#### Findings

> All 66 structured findings with F-IDs, status, presentation flags and action notes  
> → **[`03_analysis_7-findings.ipynb`](03_analysis_7-findings.ipynb)**

### KPI Overview


> Stand: Projekt abgeschlossen — Analyse + Modellierung vollständig.  
> ✅ = beantwortet · ⚠️ = teilweise / Näherungswert

---

#### Baseline KPIs

| KPI | Wert | Quelle |
|:---|:---|:---|
| OTP (arrival_delay ≤ 120s) | 87 | F-TARGET — Schwellwert ±120s = VBZ-Standard |
| Ø arrival_delay (2025 Jan–Okt, bereinigt) | 55.8s | F-TARGET-10 |
| Ø delay_delta (2025 Jan–Okt, bereinigt) | ~+5.1s | F-TARGET-09 |
| Cancellation Rate (effektiv, ab Jul 2024) | 6.2% | F-TARGET-05 |
| November-Anomalie (Artefakt ohne Bereinigung) | +17s / +26s delta | F-TARGET-06 |
| Aufwärtstrend delay_delta 2023→2025 | +4.6s → +5.1s → +5.1s | F-TARGET-09 |

#### Model KPIs

| Modell | Features | Test MAE | vs. Baseline | Quelle |
|:---|:---|:---:|:---:|:---|
| Stop Mean Baseline | — | 50.0s | — | `06_prediction_1` |
| LightGBM v1 | 32 | 45.7s | −4.3s | `06_prediction_2` |
| **LightGBM v2** | **34** | **18,56 s** | **−31.4s (−63%)** | `06_prediction_4` |
| XGBoost (Robustness) | 34 | vergleichbar | kein Vorteil | `06_prediction_5` |

---

#### Q1 — Where do delays occur?

**✅ Beantwortet.**

| Teilfrage | Status | Findings |
|:---|:---|:---|
| Welche Haltestellen sind die grössten Hotspots? | ✅ Periphere Aussenkorridore: Friedhof Enzenbühl 93.8s, Balgrist 85.2s, Leutschenbach 82.7s — NICHT zentrale Knotenpunkte | F-SPAT-01, F-SPAT-07 |
| Welche Linien akkumulieren Verspätung? | ✅ L11 (68.7s, OTP 82%) kritischste Hauptlinie; alle Linien akkumulieren positiv | F-SPAT-04, F-SPAT-05 |
| Welche Stadtkreise haben die höchsten Delays? | ✅ Kreis 11 (68,3 s, OTP 83%) schlechtester; Kreis 12 (66.3s); Kreis 5 (49.9s, OTP 89%) bester | F-SPAT-03 |
| Sind Liniendichte und Verspätung korreliert? | ✅ Keine Korrelation — 0 Overlap; Haldenegg (15 Linien, 44.5s) und Paradeplatz (14 Linien, 48.2s) unter Netzschnitt | F-SPAT-07, F-NET-05 |

---

#### Q2 — When do delays occur?

**✅ Beantwortet.**

| Teilfrage | Status | Findings |
|:---|:---|:---|
| Welche Tagesstunden sind kritisch? | ✅ Kein klassischer Morgenrush (7h=48.9s unter Ø); Peak 21h=67.9s (Events-Abreisewelle); Abend-Peak 17h=65.2s | F-TEMP-01 |
| Welcher Wochentag ist am schlechtesten? | ✅ Donnerstag (60.4s, P95=194s); Montag (52.3s) und Sonntag (48.4s) beste Tage | F-TEMP-02 |
| Welcher Monat ist am schlechtesten? | ✅ November (Nov 2023=67.9s, Nov 2024=72.9s — jeweils Jahreshöchstwert) | F-TEMP-05 |
| Welche Jahreszeit ist am schlechtesten? | ✅ Herbst (61.2s, OTP 85.2%); Winter überraschend beste Jahreszeit (51.7s, OTP 88.9%) | F-TEMP-06 |
| Gibt es einen Aufwärtstrend? | ✅ Ja, moderat; 2024 Frühling/Sommer +4–7s über 2023; 2025 Stabilisierung | F-TARGET-09, F-TEMP-07 |
| Ist der Schulferien-Effekt messbar? | ⚠️ Im Rolling-Average sichtbar, nicht separat quantifiziert | F-TEMP-08 |

---

#### Q3 — What amplifies delays?

**✅ Beantwortet.**

| Einflussfaktor | Effekt | Status | Findings |
|:---|:---|:---|:---|
| Schnee | stark positiv (+54.0s, OTP −10.9pp) | ✅ stärkster Wettereffekt | F-WEAT-01 |
| Starkregen | moderat positiv (+23.3s); skaliert mit Intensität | ✅ Dosis-Wirkungs-Effekt | F-WEAT-02 |
| Wind | minimal | ✅ `is_windy` = 100% NaN — aus Feature-Set entfernt | F-WEAT-03 |
| Hohe Temperatur (>20°C) | schwach positiv (+2.0s); Kälte (0–5°C) beste Bedingung (53.8s) | ✅ Frost-Hypothese falsch | F-WEAT-04 |
| Feiertag | stark negativ (−9.9s, OTP +3.6pp) — bester Tag-Typ | ✅ Berufsverkehr-Reduktion überwiegt | F-EVNT-01 |
| Grosse Events | positiv (+10.5s), primär Abend-Phänomen (18–22h) | ✅ Fachmessen schlechteste Kategorie (66.0s) | F-EVNT-03, F-EVNT-04 |
| Autoverkehr (MIV) | messbar — Ferien/Feiertage deutlich besser | ✅ Montag Homeoffice-Effekt bestätigt | F-TEMP-02, F-TEMP-06 |

---

#### Q4 — What does the target look like?

**✅ Beantwortet.**

| Aspekt | Antwort | Findings |
|:---|:---|:---|
| Verteilungsform | Rechtsschiefe (Long Tail) — LightGBM robust, kein Log-Transform nötig | F-TARGET-01 |
| Bimodalität | delay_delta bimodal — Recovery-Cluster ~−45s und Akkumulations-Cluster ~+15s | F-TARGET-02 |
| Datenqualität Cancellations | Artefakt vor Jul 2024 — Datendefinitions-Änderung; effektive Rate 6.2% | F-TARGET-05, F-TARGET-11 |
| Datenqualität Nov/Dez 2025 | GTFS-Artefakt (+17.1s/+26.0s delta) — aus Train+Test entfernt | F-TARGET-06 |
| Linie E Outlier | OTP 55.7%, Ø 130s — aus lf_clean entfernt, separat behandelt | F-TARGET-12, F-NET-08 |

---

#### Q5 — Which features correlate most strongly?

**✅ Beantwortet — Feature Importance aus LightGBM v1 (Gain).**

| Feature-Gruppe | Stärke | Bestätigt durch |
|:---|:---|:---|
| `dwell_time` | ⭐⭐⭐ #1 (Gain 14.8M) — aber konfundiert (F-SIM-01/02) | LightGBM v1 Feature Importance |
| `stop_name` | ⭐⭐⭐ #2 (Gain 12.7M) | LightGBM v1 Feature Importance |
| `prev_trip_delay` | ⭐⭐⭐ #1 in v2 — Kaskadenindikator, stärkstes neues Feature | LightGBM v2 (F-NET-07) |
| `hour` | ⭐⭐⭐ hoch — konsistentester temporaler Effekt | F-TEMP-01 |
| `line_name` | ⭐⭐⭐ hoch | F-SPAT-05 |
| `day_of_week` | ⭐⭐ mittel | F-TEMP-02 |
| `has_snow` | ⭐⭐ mittel (saisonal) | F-WEAT-01 |
| `is_holiday` | ⭐⭐ mittel (negativ) | F-EVNT-01 |
| `event_weight × hour` | ⭐⭐ mittel (Abend) | F-EVNT-03 |
| `month` / `season` | ⭐⭐ mittel | F-TEMP-05/06 |
| `precipitation` | ⭐ gering–mittel | F-WEAT-02 |
| ~~`is_windy`~~ | ❌ entfernt — 100% NaN | F-WEAT-03 |

---

#### Q6 — Are delays predictable?

**✅ Ja — LightGBM v2 MAE 18,56 s, −63% gegenüber Baseline.**

| Aspekt | Ergebnis | Quelle |
|:---|:---|:---|
| Vorhersagbarkeit | ✅ Bestätigt — starke zeitliche + räumliche Muster, Modell schlägt Baseline deutlich | `06_prediction_*` |
| Feature Engineering | ✅ 48 Features exportiert · `is_windy` entfernt (F-WEAT-03) | `05_feature_engineering` |
| Bekannte Schwierigkeiten | Extremwerte robust durch LightGBM behandelt; Events unbalanced aber kein Problem | F-TARGET-07, F-EVNT-05 |
| Baseline | Stop Mean MAE 50.0s | `06_prediction_1` |
| LightGBM v1 | MAE 45.7s · MBE +8,3 s · 32 Features | `06_prediction_2/3` |
| **LightGBM v2** | **MAE 18,56 s · −63% · `prev_trip_delay` als Kaskadenindikator** | `06_prediction_4` |
| XGBoost Robustness | Vergleichbar, kein Vorteil — LightGBM als Produktionsmodell bestätigt | `06_prediction_5` |
| Schlüsselerkenntnis | `prev_trip_delay` (F-NET-07) transformiert eine Analyse-Erkenntnis in ein Modell-Signal — Vorhersagbar = Strukturell = Steuerbar | F-REC-01 |


### Modelling Insights


### Feature Priority — Confirmed Results

The 66 findings were prioritized by expected predictive value before model training. The ranking was confirmed by LightGBM feature importance — with one unexpected addition.

#### Priority 1 — Core Features (structural effects)

- **Temporal features** proved the strongest signals overall:
  - **Hour of day:** No morning rush — peak at 21h (event departure waves), 17h (commute peak).
  - **Weekday:** Thursday worst (60.4s, P95=194s); Sunday and Monday benefit from reduced car traffic.
  - **Month/Season:** November peak, Winter best — counterintuitively: reduced MIV outweighs snow.
- **Line:** L11 (68.7s), L8 structurally above average. Line E excluded as outlier (130s, OTP 55.7%) — F-TARGET-12.
- **stop_sequence:** First stops systematically biased — removed from lf_clean; used as indicator feature.

#### Priority 2 — Important, measurable

- **Weather features:**
  - **Snow:** Strongest single effect (+54s, OTP −10.9pp) — geographically distinct from rain (elevation vs. river valleys).
  - **Heavy rain:** +23.3s; dose-response confirmed.
  - **is_hot (>20°C):** Small but consistent effect (+2.0s).
- **Stop identity:** Peripheral stops chronically delayed (Friedhof Enzenbühl, Balgrist) — target encoding by stop name effective.
- **District:** K11/K12 elevated; K5 best despite central density.

#### Priority 3 — Useful, with caveats

- **Events & holidays:** Large events +10.5s (evening only, 18–22h). Holidays reduce delay −9.9s — counterintuitive but robust across all 3 years.
- **gtfs_year / schedule change:** Weak signal network-wide (+0.5s) — confirmed by feature importance.
- **delay_delta bimodality:** Real network pattern, not a terminus artifact. `prev_trip_delay` (F-NET-07) operationalised this as the cascade indicator.

---

### What Proved Most Predictive

- **Temporal stability confirmed:** a line that underperformed in 2023 remained below average in 2024 and 2025, with consistent hour-of-day and weekday patterns. The temporal × spatial combination formed the model's foundation.
- **Nov/Dec 2025 GTFS artifact excluded** — removing these rows prevented a +17–26s delta distortion in model residuals (F-TARGET-06). ✅
- **Unexpected breakthrough:** `prev_trip_delay` (cascade indicator, F-NET-07) became the strongest new feature in v2. The jump from MAE 45.7s to 18,56 s (−63%) came not from algorithm tuning, but from translating one analysis finding — Pearson r ≥ 0.85 between consecutive stops — directly into a prediction signal.

---

### Feature Priority Summary

| Priority | Feature Group | Outcome |
| :--- | :--- | :--- |
| **1** | Hour × Weekday × Month | ✅ Confirmed — largest variance share, stable across all 3 years |
| **1** | Line | ✅ Confirmed — L11/L8 structurally worse; Line E excluded |
| **2** | Weather (Snow > Rain > Heat) | ✅ Confirmed — snow +54s strongest single effect |
| **2** | Stop identity / District | ✅ Confirmed — peripheral stops chronically high |
| **3** | Events / Holidays | ✅ Confirmed — holidays reduce delay (counterintuitive but robust) |
| **3** | gtfs_year / Schedule change | ✅ Confirmed weak — low feature importance as expected |
| **+** | `prev_trip_delay` (cascade) | ✅ **Unexpected breakthrough** — strongest new feature in v2 (F-NET-07) |


---

## 🔬 Research Opportunities & Future Questions

**Kontext:** Während die 6 Kernfindings die Analyse-Hauptfragen beantworten, offenbarte die interaktive Erkundung des Dashboards **weitere systematische Muster**, die tiefere Analysen rechtfertigen könnten.

Diese Sektion dokumentiert **Beobachtungen und Hypothesen** die **nicht Teil dieser Analyse sind**, aber als **strukturiertes Forschungs-Backlog** dienen können.

---

### Why Dashboard Exploration Matters

Das Projekt produziert nicht nur **statische Findings** (F1–F6), sondern auch ein **interaktives Dashboard** (`apps/dashboard/app.py`). Beim manuellen Durchklicken der 16 Linien und Vergleichen verschiedener Filter-Kombinationen entstehen neue Fragen, die eine Komplett-Analyse nicht natürlich evoziert:

- **"Warum ist Linie 11 in Richtung A 10s schneller als Richtung B?"** ← beobachtet beim Direction Filter
- **"Welche Linien profitieren am meisten vom Wochenende?"** ← beobachtet bei Line Comparison
- **"Gibt es Linien, die Delays absorbieren statt zu verstärken?"** ← beobachtet bei Cascade Pattern Exploration

Diese Ad-hoc-Entdeckungen sind **Signals für strukturelle Potenziale** — nicht genug für einen Hauptfinding, aber genug um neue Analysen zu motivieren.

---

### Research Opportunities — Structured Backlog

Detaillierte Hypothesen, Implementation Paths und Prioritäten:

**→ Siehe [BACKLOG.md — Research Opportunities Section](../BACKLOG.md#-research-opportunities--dashboard-discovery)**

| OP # | Beobachtung | Prio | Next Step |
|:---|:---|:---|:---|
| **OP-1** | Direction-Asymmetrie (~10s Delta zwischen Richtung A/B bei L11) | 2 | Neue Analyse: Direction-spezifische Hotspots |
| **OP-2** | Stop-Variabilität (manche Stops stabil, manche chaotisch; std > 100s) | 3 | Cluster-Analyse: Stop-Typen nach Varianz-Profil |
| **OP-3** | Linienlänge ↔ Delay nicht-linear (lange Linien schlechter, aber nicht proportional) | 2 | Scatter-Plot: n_stops vs. Ø Delay pro Linie |
| **OP-4** | Weekend != Weekday pro Linie (manche besser, manche schlechter) | 2 | Heatmap: Line × DayType Interaction |
| **OP-5** | Wetter-Empfindlichkeit unterschiedlich (L11 Schnee-sensitiv, L2 nicht) | 3 | Topografie-Analyse: Höhenprofil × Weather-Sensitivity |
| **OP-6** | Schedule Bias (L13 immer früh, L7 immer spät) | 2 | Analyse: Schedule Margin Calculation per Line |
| **OP-7** | Kaskaden-Verstärker vs. -Dämpfer (Linien verhalten sich unterschiedlich) | 1 | Neue Notebook: Cascade Mechanics Stratification |

---

### Decision Framework

**Nach diesem Projekt können die Opportunities priorisiert werden nach:**

1. **Signifikanz:** Sind die beobachteten Unterschiede > 5% der Gesamtvarianz?
2. **Actionability:** Können die Erkenntnisse zu konkreten Maßnahmen führen?
3. **Effort:** Passt die Analyse in eine Normal-Session (1–2h)?

**Beispiel — OP-1 (Direction-Asymmetrie):**
- Signifikant? Zu klären (aktuell nur L11 beobachtet; systematisch oder Zufall?)
- Actionable? JA — wenn asymmetrisch, dann Scheduling-Ansatz pro Richtung
- Effort? Mittel — würde einen Daten-Pipeline-Umbau erfordern (siehe `DIRECTION_ID_ARCHITECTURE_PLAN.md`)

---

### Next Steps

1. **Während Dashboard-Exploration:** Neue Beobachtungen direkt in BACKLOG "Sammlung nach Session" notieren
2. **Nach Session:** OP-Prioritäten diskutieren und ggf. neue Analyse-Notebooks starten
3. **Langfristig:** Opportunities als **systematischer Feedback-Loop** nutzen um die Analysen iterativ zu verfeinern

